<h2>Xây dựng mô hình dự đoán kết quả các trận đấu</h2>


Phần này sẽ kết hợp các kỹ thuật và phương pháp trong Machine Learning để xây dựng một mô hình dự đoán kết quả của các trận đấu trong tương lai, sử dụng Spark Machine Learning

<h2>I. Các thư viện và model được sử dụng</h2>

In [66]:
!pip install pyspark

In [67]:
import pandas as pd
import matplotlib.pyplot as plt

#import findspark
#import pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id, lit, when

from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, NaiveBayes, RandomForestClassifier
from pyspark.ml.feature import OneHotEncoder,StringIndexer,VectorAssembler
#from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.ml.stat import ChiSquareTest
from pyspark.sql.types import *

from google.cloud import bigquery
from google.cloud import storage
#from google.colab import auth

import os

Sử dụng Authentication nếu chạy ở local.

In [68]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./keys/key.json"

<h2>II. Dataset Model</h2>

Chọn tên project

In [69]:
project_id = 'second-chariot-420108'
bigQuery_client = bigquery.Client(project=project_id)
storage_client = storage.Client()

In [70]:
query_factMatchStatistics = """
SELECT * FROM `second-chariot-420108.Football_DataWarehouse_1.Fact_Match_Statistics`
"""
query_dimMatch = """
SELECT * FROM `second-chariot-420108.Football_DataWarehouse_1.Dim_Match`
"""

query_dimTeam = """
SELECT * FROM `second-chariot-420108.Football_DataWarehouse_1.Dim_Team`
"""

In [71]:
match_dataset_model = pd.read_gbq(query_factMatchStatistics, project_id=project_id, dialect='standard')
dim_match = pd.read_gbq(query_dimMatch, project_id=project_id, dialect='standard')
dim_team = pd.read_gbq(query_dimTeam, project_id=project_id, dialect='standard')

In [72]:
match_dataset_model = match_dataset_model.rename(columns={'Match': 'Match_Key'})

In [73]:
match_dataset_model = match_dataset_model.drop(['Home_Total_Players_Stats', 'Home_Minutes',
                                                'Away_Total_Players_Stats', 'Away_Minutes'], axis=1)

In [74]:
match_dataset_model
match_dataset_model.to_csv('./match_dataset_model.csv', index=False)

In [75]:
match_dataset_model_desc = pd.merge(match_dataset_model, dim_match[['Match_Key','Match_Date']], on='Match_Key', how='inner')

In [76]:
match_dataset_model_desc = match_dataset_model_desc.sort_values(by='Match_Date', ascending=False).reset_index(drop=True)
match_dataset_model_desc

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Away_Succ_Take_Ons,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date
0,12646,136,33,18,4,17,5,15,3,9,...,9,72.000000,71.533333,70.083333,77.100000,74.900000,77.266667,2,1,20240526
1,7846,64,39,11,6,24,4,16,1,7,...,8,77.153846,77.812500,74.363636,78.142857,76.800000,77.785714,1,2,20240526
2,7845,62,54,18,2,14,5,14,2,12,...,13,76.200000,74.750000,73.846154,69.600000,73.000000,71.636364,2,2,20240526
3,12643,118,40,18,2,11,11,22,1,7,...,8,78.250000,76.411765,72.692308,73.181818,73.866667,73.285714,3,0,20240526
4,12648,117,35,17,9,21,6,3,2,5,...,6,71.000000,69.650000,71.500000,81.600000,79.350000,80.615385,2,2,20240526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,2592,35,38,9,2,23,21,13,2,5,...,17,75.500000,68.727273,72.625000,84.857143,84.000000,81.625000,0,3,20180811
10679,387,9,57,20,5,16,21,16,1,6,...,8,71.545455,72.000000,73.000000,0.000000,0.000000,0.000000,3,4,20180811
10680,2590,33,63,11,7,11,32,40,0,7,...,9,77.500000,75.000000,74.700000,0.000000,0.000000,0.000000,2,0,20180811
10681,381,10,60,7,5,20,19,14,1,7,...,11,78.285714,77.800000,75.333333,73.000000,71.833333,72.571429,4,0,20180810


In [77]:
match_dataset_model_desc.dtypes

Match_Key            Int64
Home_Team            Int64
Home_Possession      Int64
Home_Fouls           Int64
Home_Corners         Int64
                    ...   
Away_Midfield      float64
Away_Defense       float64
Home_Score           Int64
Away_Score           Int64
Match_Date           Int64
Length: 82, dtype: object

In [78]:
df_without_nan_all = match_dataset_model_desc
df_without_nan_all = df_without_nan_all.dropna()
df_without_nan_all

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Away_Succ_Take_Ons,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date
0,12646,136,33,18,4,17,5,15,3,9,...,9,72.000000,71.533333,70.083333,77.100000,74.900000,77.266667,2,1,20240526
1,7846,64,39,11,6,24,4,16,1,7,...,8,77.153846,77.812500,74.363636,78.142857,76.800000,77.785714,1,2,20240526
2,7845,62,54,18,2,14,5,14,2,12,...,13,76.200000,74.750000,73.846154,69.600000,73.000000,71.636364,2,2,20240526
3,12643,118,40,18,2,11,11,22,1,7,...,8,78.250000,76.411765,72.692308,73.181818,73.866667,73.285714,3,0,20240526
4,12648,117,35,17,9,21,6,3,2,5,...,6,71.000000,69.650000,71.500000,81.600000,79.350000,80.615385,2,2,20240526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,2592,35,38,9,2,23,21,13,2,5,...,17,75.500000,68.727273,72.625000,84.857143,84.000000,81.625000,0,3,20180811
10679,387,9,57,20,5,16,21,16,1,6,...,8,71.545455,72.000000,73.000000,0.000000,0.000000,0.000000,3,4,20180811
10680,2590,33,63,11,7,11,32,40,0,7,...,9,77.500000,75.000000,74.700000,0.000000,0.000000,0.000000,2,0,20180811
10681,381,10,60,7,5,20,19,14,1,7,...,11,78.285714,77.800000,75.333333,73.000000,71.833333,72.571429,4,0,20180810


In [79]:
match_dataset_model_desc[~match_dataset_model_desc['Match_Key'].isin(df_without_nan_all['Match_Key'])]['Match_Key'].tolist()

[]

In [80]:
df_without_nan = match_dataset_model_desc[['Home_Score', 'Away_Score']]
df_without_nan = df_without_nan.dropna()
df_without_nan

,Home_Score,Away_Score
0,2,1
1,1,2
2,2,2
3,3,0
4,2,2
...,...,...
10678,0,3
10679,3,4
10680,2,0
10681,4,0


In [81]:
len(match_dataset_model_desc) - len(df_without_nan)

0

Với mỗi cột, nếu kiểu dữ liệu là Int64 thì ép kiểu thành Float

Home_Team và Away_Team ép kiểu thành str.

In [82]:

for col in match_dataset_model_desc.columns:
    if col != 'Match_Key' and col != 'Home_Team' and col != 'Away_Team' and match_dataset_model_desc[col].dtype == 'Int64':
        match_dataset_model_desc[col] = match_dataset_model_desc[col].astype(float)

match_dataset_model_desc['Home_Team'] = match_dataset_model_desc['Home_Team'].astype(int)
match_dataset_model_desc['Home_Team'] = match_dataset_model_desc['Home_Team'].astype(str)

match_dataset_model_desc['Away_Team'] = match_dataset_model_desc['Away_Team'].astype(int)
match_dataset_model_desc['Away_Team'] = match_dataset_model_desc['Away_Team'].astype(str)

In [83]:
match_dataset_model_desc.dtypes

Match_Key            Int64
Home_Team           object
Home_Possession    float64
Home_Fouls         float64
Home_Corners       float64
                    ...   
Away_Midfield      float64
Away_Defense       float64
Home_Score         float64
Away_Score         float64
Match_Date         float64
Length: 82, dtype: object

Danh sách các đội:

In [84]:
list_all_teams = match_dataset_model_desc['Home_Team'].unique()
list_all_teams

array(['136', '64', '62', '118', '117', '134', '138', '74', '72', '124',
       '116', '85', '88', '131', '80', '77', '79', '130', '63', '126',
       '121', '143', '119', '21', '144', '8', '20', '3', '57', '65',
       '128', '76', '83', '29', '86', '39', '125', '25', '5', '55', '48',
       '71', '120', '49', '40', '66', '61', '42', '78', '24', '1', '38',
       '43', '84', '51', '115', '135', '70', '109', '140', '92', '101',
       '105', '102', '93', '100', '99', '129', '14', '30', '44', '52',
       '6', '114', '89', '19', '10', '112', '26', '12', '11', '2', '104',
       '97', '36', '33', '56', '31', '47', '32', '96', '46', '106', '91',
       '23', '103', '87', '75', '81', '145', '7', '28', '27', '41', '54',
       '37', '9', '141', '122', '95', '90', '50', '142', '68', '34', '4',
       '111', '15', '113', '123', '73', '82', '53', '137', '22', '132',
       '16', '133', '139', '60', '107', '110', '13', '18', '127', '17',
       '98', '35', '45', '108'], dtype=object)

Thay đổi các statistic

In [85]:
# For each team, get all the matches that the team played
# Then calculate the average of the last 2 matches for each feature
# Then update the match_dataset_model_desc with the new values
for team in list_all_teams:
    print(team)

    this_team = match_dataset_model_desc[(match_dataset_model_desc['Home_Team'] == team) |
                (match_dataset_model_desc['Away_Team'] == team)]
    for i in range(0, len(this_team)-2):

        if(this_team.iloc[i,this_team.columns.get_loc('Home_Team')] == team):

            for j in range(2, 37):

                list_2_elements = []

                for k in range(i+1, i+3):
                    if(this_team.iloc[k,this_team.columns.get_loc('Home_Team')] == team):

                        list_2_elements.append(this_team.iloc[k, j])

                    if(this_team.iloc[k,this_team.columns.get_loc('Away_Team')] == team):

                        list_2_elements.append(this_team.iloc[k, j+36])

                match_key = this_team.iloc[i,this_team.columns.get_loc('Match_Key')]


                match_dataset_model_desc.loc[match_dataset_model_desc['Match_Key'] == match_key,
                                             match_dataset_model_desc.columns[j]] = sum(list_2_elements) / 2

        if(this_team.iloc[i,this_team.columns.get_loc('Away_Team')] == team):

            for j in range(38, 73):

                list_2_elements = []

                for k in range(i+1, i+3):
                    if(this_team.iloc[k,this_team.columns.get_loc('Home_Team')] == team):

                        list_2_elements.append(this_team.iloc[k, j-36])

                    if(this_team.iloc[k,this_team.columns.get_loc('Away_Team')] == team):

                        list_2_elements.append(this_team.iloc[k, j])

                match_key = this_team.iloc[i,this_team.columns.get_loc('Match_Key')]

                match_dataset_model_desc.loc[match_dataset_model_desc['Match_Key'] == match_key,
                                             match_dataset_model_desc.columns[j]] = sum(list_2_elements) / 2

136
64
62
118
117
134
138
74
72
124
116
85
88
131
80
77
79
130
63
126
121
143
119
21
144
8
20
3
57
65
128
76
83
29
86
39
125
25
5
55
48
71
120
49
40
66
61
42
78
24
1
38
43
84
51
115
135
70
109
140
92
101
105
102
93
100
99
129
14
30
44
52
6
114
89
19
10
112
26
12
11
2
104
97
36
33
56
31
47
32
96
46
106
91
23
103
87
75
81
145
7
28
27
41
54
37
9
141
122
95
90
50
142
68
34
4
111
15
113
123
73
82
53
137
22
132
16
133
139
60
107
110
13
18
127
17
98
35
45
108


In [86]:
match_dataset_model_desc

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Away_Succ_Take_Ons,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date
0,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,6.5,72.000000,71.533333,70.083333,77.100000,74.900000,77.266667,2.0,1.0,20240526.0
1,7846,64,51.5,9.0,2.0,20.0,12.0,23.0,0.0,10.0,...,11.5,77.153846,77.812500,74.363636,78.142857,76.800000,77.785714,1.0,2.0,20240526.0
2,7845,62,52.0,13.5,2.0,8.5,14.5,20.0,2.5,8.5,...,4.0,76.200000,74.750000,73.846154,69.600000,73.000000,71.636364,2.0,2.0,20240526.0
3,12643,118,57.0,15.5,5.5,15.5,14.0,13.0,1.0,6.0,...,5.5,78.250000,76.411765,72.692308,73.181818,73.866667,73.285714,3.0,0.0,20240526.0
4,12648,117,45.0,19.0,6.5,24.0,24.5,13.0,1.5,4.5,...,4.0,71.000000,69.650000,71.500000,81.600000,79.350000,80.615385,2.0,2.0,20240526.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,2592,35,38.0,9.0,2.0,23.0,21.0,13.0,2.0,5.0,...,17.0,75.500000,68.727273,72.625000,84.857143,84.000000,81.625000,0.0,3.0,20180811.0
10679,387,9,57.0,20.0,5.0,16.0,21.0,16.0,1.0,6.0,...,8.0,71.545455,72.000000,73.000000,0.000000,0.000000,0.000000,3.0,4.0,20180811.0
10680,2590,33,63.0,11.0,7.0,11.0,32.0,40.0,0.0,7.0,...,9.0,77.500000,75.000000,74.700000,0.000000,0.000000,0.000000,2.0,0.0,20180811.0
10681,381,10,60.0,7.0,5.0,20.0,19.0,14.0,1.0,7.0,...,11.0,78.285714,77.800000,75.333333,73.000000,71.833333,72.571429,4.0,0.0,20180810.0


<h2>III. Xây dựng mô hình dự đoán kết quả trận đấu</h2>

Phần này sẽ kết hợp các kỹ thuật và phương pháp trong Machine Learning để xây dựng một mô hình dự đoán kết quả của các trận đấu trong tương lai, sử dụng Spark Machine Learning

<h3>1. Phân tích dữ liệu</h3>

In [87]:
dataset_model = match_dataset_model_desc.copy()

Thêm attribute Result thể hiện kết quả trận đấu cho đội sân nhà với 3 trường hợp: Win, Draw, Lose

In [88]:
list_result = ['Win' if home_score > away_score
               else 'Draw' if home_score == away_score
               else 'Lose' for home_score, away_score
               in zip(dataset_model['Home_Score'], dataset_model['Away_Score'])]

In [89]:
dataset_model['Result'] = list_result
dataset_model

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date,Result
0,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,72.000000,71.533333,70.083333,77.100000,74.900000,77.266667,2.0,1.0,20240526.0,Win
1,7846,64,51.5,9.0,2.0,20.0,12.0,23.0,0.0,10.0,...,77.153846,77.812500,74.363636,78.142857,76.800000,77.785714,1.0,2.0,20240526.0,Lose
2,7845,62,52.0,13.5,2.0,8.5,14.5,20.0,2.5,8.5,...,76.200000,74.750000,73.846154,69.600000,73.000000,71.636364,2.0,2.0,20240526.0,Draw
3,12643,118,57.0,15.5,5.5,15.5,14.0,13.0,1.0,6.0,...,78.250000,76.411765,72.692308,73.181818,73.866667,73.285714,3.0,0.0,20240526.0,Win
4,12648,117,45.0,19.0,6.5,24.0,24.5,13.0,1.5,4.5,...,71.000000,69.650000,71.500000,81.600000,79.350000,80.615385,2.0,2.0,20240526.0,Draw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,2592,35,38.0,9.0,2.0,23.0,21.0,13.0,2.0,5.0,...,75.500000,68.727273,72.625000,84.857143,84.000000,81.625000,0.0,3.0,20180811.0,Lose
10679,387,9,57.0,20.0,5.0,16.0,21.0,16.0,1.0,6.0,...,71.545455,72.000000,73.000000,0.000000,0.000000,0.000000,3.0,4.0,20180811.0,Lose
10680,2590,33,63.0,11.0,7.0,11.0,32.0,40.0,0.0,7.0,...,77.500000,75.000000,74.700000,0.000000,0.000000,0.000000,2.0,0.0,20180811.0,Win
10681,381,10,60.0,7.0,5.0,20.0,19.0,14.0,1.0,7.0,...,78.285714,77.800000,75.333333,73.000000,71.833333,72.571429,4.0,0.0,20180810.0,Win


Kiểm tra số row thiếu dữ liệu, số thuộc tính input từ Home_Team đến Away_Defense

In [90]:
nan_rows = dataset_model.isna().sum(axis=1)
num_nan_rows = len(nan_rows[nan_rows > 0])
print("Tổng số trận:", len(dataset_model))
print("Số trận bị thiếu dữ liệu:", num_nan_rows)
print("Số thuộc tính input:", (dataset_model.columns.get_loc("Away_Defense")
                               - dataset_model.columns.get_loc("Home_Team")) + 1)

Tổng số trận: 10683
Số trận bị thiếu dữ liệu: 0
Số thuộc tính input: 78


In [91]:
dataset_model[dataset_model.isna().any(axis=1)]

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date,Result


Xoá NaN

In [92]:
#dataset_model = dataset_model[dataset_model['Match_Key'] != 10768]
#dataset_model = dataset_model[dataset_model['Match_Key'] != 10753]

#attack_mid_defense = 0
dataset_model = dataset_model[(dataset_model['Home_Attack'] != 0) &
                              (dataset_model['Home_Midfield'] != 0) &
                              (dataset_model['Home_Defense'] != 0)]
dataset_model = dataset_model[(dataset_model['Away_Attack'] != 0) &
                              (dataset_model['Away_Midfield'] != 0) &
                              (dataset_model['Away_Defense'] != 0)]

In [93]:
print("Số trận bị thiếu dữ liệu:", len(dataset_model.isna().sum(axis=1)[dataset_model.isna().sum(axis=1) > 0]))

Số trận bị thiếu dữ liệu: 0


In [94]:
dataset_model

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date,Result
0,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,72.000000,71.533333,70.083333,77.100000,74.900000,77.266667,2.0,1.0,20240526.0,Win
1,7846,64,51.5,9.0,2.0,20.0,12.0,23.0,0.0,10.0,...,77.153846,77.812500,74.363636,78.142857,76.800000,77.785714,1.0,2.0,20240526.0,Lose
2,7845,62,52.0,13.5,2.0,8.5,14.5,20.0,2.5,8.5,...,76.200000,74.750000,73.846154,69.600000,73.000000,71.636364,2.0,2.0,20240526.0,Draw
3,12643,118,57.0,15.5,5.5,15.5,14.0,13.0,1.0,6.0,...,78.250000,76.411765,72.692308,73.181818,73.866667,73.285714,3.0,0.0,20240526.0,Win
4,12648,117,45.0,19.0,6.5,24.0,24.5,13.0,1.5,4.5,...,71.000000,69.650000,71.500000,81.600000,79.350000,80.615385,2.0,2.0,20240526.0,Draw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10674,385,4,43.0,14.0,6.0,22.0,21.0,32.0,2.0,10.0,...,74.000000,74.200000,75.750000,76.500000,73.250000,72.500000,2.0,1.0,20180811.0,Win
10677,386,8,56.0,12.0,2.0,10.0,10.0,19.0,1.0,7.0,...,69.500000,71.133333,69.166667,75.166667,72.588235,74.166667,3.0,1.0,20180811.0,Win
10678,2592,35,38.0,9.0,2.0,23.0,21.0,13.0,2.0,5.0,...,75.500000,68.727273,72.625000,84.857143,84.000000,81.625000,0.0,3.0,20180811.0,Lose
10681,381,10,60.0,7.0,5.0,20.0,19.0,14.0,1.0,7.0,...,78.285714,77.800000,75.333333,73.000000,71.833333,72.571429,4.0,0.0,20180810.0,Win


<h3>2. Kiểm tra mối tương quan, biến đổi thuộc tính</h3>

Khởi chạy một phiên làm việc với SparkSession

In [95]:
spark = (SparkSession
         .builder
         .appName("Classifications Technique")
         .getOrCreate())

Chuyển đổi Dataframe pandas sang Dataframe Spark

In [96]:
matches = spark.createDataFrame(dataset_model)
display(matches.limit(5).toPandas())

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date,Result
0,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,72.000000,71.533333,70.083333,77.100000,74.900000,77.266667,2.0,1.0,20240526.0,Win
1,7846,64,51.5,9.0,2.0,20.0,12.0,23.0,0.0,10.0,...,77.153846,77.812500,74.363636,78.142857,76.800000,77.785714,1.0,2.0,20240526.0,Lose
2,7845,62,52.0,13.5,2.0,8.5,14.5,20.0,2.5,8.5,...,76.200000,74.750000,73.846154,69.600000,73.000000,71.636364,2.0,2.0,20240526.0,Draw
3,12643,118,57.0,15.5,5.5,15.5,14.0,13.0,1.0,6.0,...,78.250000,76.411765,72.692308,73.181818,73.866667,73.285714,3.0,0.0,20240526.0,Win
4,12648,117,45.0,19.0,6.5,24.0,24.5,13.0,1.5,4.5,...,71.000000,69.650000,71.500000,81.600000,79.350000,80.615385,2.0,2.0,20240526.0,Draw


Sử dụng phương pháp One-hot Encoding để biến đổi một số thuộc tính sang numeric

In [97]:
#Attributes không dùng tới
unused_cols = ['Match_Key', 'Match_Date', 'Home_Score', 'Away_Score']
#Output
output_cols = ['Result']

- Chọn các input và các input cần được One-hot Encoding

In [98]:
input_cols = [column for column in matches.columns
              if column not in output_cols and column not in unused_cols]
encode_cols = ['Home_Team','Away_Team']

In [99]:
def EncodedData(matches, unused_cols, output_cols, encode_cols):
    #String Indexer
    indexer = StringIndexer(inputCols=encode_cols, outputCols = [encode_col+ "_Index" for encode_col in encode_cols])
    encoded_df = indexer.fit(matches).transform(matches)

    #OneHot Encoder
    encodeer = OneHotEncoder(inputCols=[encode_col+"_Index" for encode_col in encode_cols],
                             outputCols=[encode_col+"_Onehot" for encode_col in encode_cols])
    encoded_df = encodeer.fit(encoded_df).transform(encoded_df)

    #Lấy ra những cột bị Index
    indexed_cols = [encode_col +"_Index" for encode_col in encode_cols]

    #Những cột được assembled là những cột không nằm trong unused_cols và indexed_cols và encode_cols và output_cols
    vector_assembled_input_cols = [col for col in encoded_df.columns
                                            if col not in unused_cols
                                            and col not in indexed_cols
                                            and col not in encode_cols
                                            and col not in output_cols
                                            ]
    assembler = VectorAssembler(inputCols=vector_assembled_input_cols,outputCol="Features")
    encoded_df = assembler.transform(encoded_df.select('*'))
    return encoded_df

- Tập data sau quá trình One-hot Encoding

In [100]:
encoded_matches = EncodedData(matches,unused_cols,output_cols,encode_cols)
display(encoded_matches.limit(5).toPandas())

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Away_Defense,Home_Score,Away_Score,Match_Date,Result,Home_Team_Index,Away_Team_Index,Home_Team_Onehot,Away_Team_Onehot,Features
0,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,77.266667,2.0,1.0,20240526.0,Win,76.0,37.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(47.0, 16.5, 6.0, 20.0, 19.0, 20.0, 3.0, 8.0, ..."
1,7846,64,51.5,9.0,2.0,20.0,12.0,23.0,0.0,10.0,...,77.785714,1.0,2.0,20240526.0,Lose,15.0,7.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","(51.5, 9.0, 2.0, 20.0, 12.0, 23.0, 0.0, 10.0, ..."
2,7845,62,52.0,13.5,2.0,8.5,14.5,20.0,2.5,8.5,...,71.636364,2.0,2.0,20240526.0,Draw,4.0,8.0,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...","(52.0, 13.5, 2.0, 8.5, 14.5, 20.0, 2.5, 8.5, 2..."
3,12643,118,57.0,15.5,5.5,15.5,14.0,13.0,1.0,6.0,...,73.285714,3.0,0.0,20240526.0,Win,23.0,39.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(57.0, 15.5, 5.5, 15.5, 14.0, 13.0, 1.0, 6.0, ..."
4,12648,117,45.0,19.0,6.5,24.0,24.5,13.0,1.5,4.5,...,80.615385,2.0,2.0,20240526.0,Draw,54.0,19.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(45.0, 19.0, 6.5, 24.0, 24.5, 13.0, 1.5, 4.5, ..."


Thêm attribute Label ứng với 3 nhãn Result: Win (0.0), Lose (1.0), Draw (2.0)

In [101]:
class_indexer = StringIndexer(inputCol = 'Result', outputCol = 'Label')

encoded_matches_label = class_indexer.fit(encoded_matches).transform(encoded_matches)

display(encoded_matches_label.limit(5).toPandas())

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Score,Away_Score,Match_Date,Result,Home_Team_Index,Away_Team_Index,Home_Team_Onehot,Away_Team_Onehot,Features,Label
0,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,2.0,1.0,20240526.0,Win,76.0,37.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(47.0, 16.5, 6.0, 20.0, 19.0, 20.0, 3.0, 8.0, ...",0.0
1,7846,64,51.5,9.0,2.0,20.0,12.0,23.0,0.0,10.0,...,1.0,2.0,20240526.0,Lose,15.0,7.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","(51.5, 9.0, 2.0, 20.0, 12.0, 23.0, 0.0, 10.0, ...",1.0
2,7845,62,52.0,13.5,2.0,8.5,14.5,20.0,2.5,8.5,...,2.0,2.0,20240526.0,Draw,4.0,8.0,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...","(52.0, 13.5, 2.0, 8.5, 14.5, 20.0, 2.5, 8.5, 2...",2.0
3,12643,118,57.0,15.5,5.5,15.5,14.0,13.0,1.0,6.0,...,3.0,0.0,20240526.0,Win,23.0,39.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(57.0, 15.5, 5.5, 15.5, 14.0, 13.0, 1.0, 6.0, ...",0.0
4,12648,117,45.0,19.0,6.5,24.0,24.5,13.0,1.5,4.5,...,2.0,2.0,20240526.0,Draw,54.0,19.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(45.0, 19.0, 6.5, 24.0, 24.5, 13.0, 1.5, 4.5, ...",2.0


<h3>4. Xây dựng mô hình dự đoán với phương pháp Logistic Regression</h3>

Tập dữ liệu có *n* cột, với mỗi lần

Các function hỗ trợ cho việc xây dựng tập train, test

- Function Reduce_Matches() loại bỏ 20 trận đầu cho tập dữ liệu với mỗi lần gọi

In [102]:
def Reduce_Matches(matches_df):

    matches_df = matches_df.withColumn("id", monotonically_increasing_id())

    filtered_data = matches_df.filter(matches_df.id >= 20)
    filtered_data = filtered_data.drop("id")

    return filtered_data

- Function Train_Dataset() trả về tập train
- Funtion Test_Dataset() trả về tập test với 10 trận mới nhất (20 row đầu tiên)

In [103]:
def Train_Dataset(matches_df):

    test_count = 20

    encoded_test_matches = matches_df.limit(test_count)
    encoded_train_matches = matches_df.subtract(encoded_test_matches)

    return encoded_train_matches

def Test_Dataset(matches_df):

    test_count = 20

    encoded_test_matches = matches_df.limit(test_count)

    return encoded_test_matches

Funtion hỗ trợ các phương pháp đánh giá cho model

In [104]:
def Measure_Function(predictions, measure):

    evaluator = MulticlassClassificationEvaluator(labelCol="Label")

    measure_method = evaluator.evaluate(predictions, {evaluator.metricName: measure})

    return measure_method

    #print("Accuracy = %g" % accuracy)
    #print("Test Error = %g" % (1.0 - accuracy))

<h4>Logistic Regression</h4>

In [105]:
#Xây dựng model dựa vào tập train
def LogisticRegression_Func(X_train):
    logit = LogisticRegression(featuresCol = "Features", labelCol = "Label", elasticNetParam=0.1, regParam=0.1)
    logitModel = logit.fit(X_train)

    return logitModel

In [106]:
#Áp dụng model trên tập test
def LogitRegres_Predictions(logitModel, X_test):

    predictions = logitModel.transform(X_test)

    return predictions

In [107]:
#
matches_df = encoded_matches_label
X_test = matches_df.filter((matches_df["Match_Key"] == 4805) | (matches_df["Match_Key"] == 4800)
| (matches_df["Match_Key"] == 4801) | (matches_df["Match_Key"] == 4806) | (matches_df["Match_Key"] == 4804)
| (matches_df["Match_Key"] == 4799) | (matches_df["Match_Key"] == 4803) | (matches_df["Match_Key"] == 4798)
| (matches_df["Match_Key"] == 4802) | (matches_df["Match_Key"] == 4797) |
(matches_df["Match_Key"] == 7841) | (matches_df["Match_Key"] == 7840)
| (matches_df["Match_Key"] == 7837) | (matches_df["Match_Key"] == 7846) | (matches_df["Match_Key"] == 7842)
| (matches_df["Match_Key"] == 7844) | (matches_df["Match_Key"] == 7843) | (matches_df["Match_Key"] == 7838)
| (matches_df["Match_Key"] == 7845) | (matches_df["Match_Key"] == 7839)|
(matches_df["Match_Key"] == 12641) | (matches_df["Match_Key"] == 12643)
| (matches_df["Match_Key"] == 12644) | (matches_df["Match_Key"] == 12648) | (matches_df["Match_Key"] == 12645)
| (matches_df["Match_Key"] == 12639) | (matches_df["Match_Key"] == 12640) | (matches_df["Match_Key"] == 12642)
| (matches_df["Match_Key"] == 12646) | (matches_df["Match_Key"] == 12647) |
(matches_df["Match_Key"] == 2531) | (matches_df["Match_Key"] == 2528)
| (matches_df["Match_Key"] == 2526) | (matches_df["Match_Key"] == 2532) | (matches_df["Match_Key"] == 2524)
| (matches_df["Match_Key"] == 2527) | (matches_df["Match_Key"] == 2530) | (matches_df["Match_Key"] == 2529)
| (matches_df["Match_Key"] == 2525) |
(matches_df["Match_Key"] == 9991) | (matches_df["Match_Key"] == 9990)
| (matches_df["Match_Key"] == 9992) | (matches_df["Match_Key"] == 9987) | (matches_df["Match_Key"] == 9989)
| (matches_df["Match_Key"] == 9994) | (matches_df["Match_Key"] == 9988) | (matches_df["Match_Key"] == 9986)
| (matches_df["Match_Key"] == 9993))

In [108]:
X_test.count()

48

In [109]:
X_test.show()

+---------+---------+---------------+----------+------------+------------+----------------+---------------+-------------+---------------+--------------+---------------+--------+--------+-------+-----------+-------+--------+---------+---------+------------+--------+--------+-----------+-------------------+-------------------+--------+--------+--------+---------------+---------------+-----------------------+----------------+--------------------+-----------------+-----------------+------------------+---------+---------------+----------+------------+------------+----------------+---------------+-------------+---------------+--------------+---------------+--------+--------+-------+-----------+-------+--------+---------+---------+------------+--------+--------+-----------+------------------+------------------+-------------------+--------+--------+---------------+---------------+-----------------------+----------------+--------------------+-----------------+-----------------+-----------------

In [110]:
X_train = matches_df.subtract(X_test)
X_train = X_train.select('Features', 'Label')
X_test = X_test.select('Features', 'Label')
X_train.count()

9061

In [111]:
logitModel = LogisticRegression_Func(X_train)
predictions = LogitRegres_Predictions(logitModel, X_test)

In [112]:
predictions.show()

+--------------------+-----+--------------------+--------------------+----------+
|            Features|Label|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|(346,[0,1,2,3,4,5...|  0.0|[1.39906848223745...|[0.27514716791886...|       1.0|
|(346,[0,1,2,3,4,5...|  1.0|[1.80742129401867...|[0.34094513894952...|       1.0|
|(346,[0,1,2,3,4,5...|  2.0|[2.08540435878205...|[0.51331050907541...|       0.0|
|(346,[0,1,2,3,4,5...|  0.0|[2.37943376514898...|[0.58119036479050...|       0.0|
|(346,[0,1,2,3,4,5...|  2.0|[1.19269089099121...|[0.16993194538473...|       1.0|
|(346,[0,1,2,3,4,5...|  2.0|[2.55497443173884...|[0.64726460781802...|       0.0|
|(346,[0,1,2,3,4,5...|  1.0|[1.57135682143536...|[0.35088749938836...|       0.0|
|(346,[0,1,2,3,4,5...|  2.0|[1.84558070778811...|[0.41583078301967...|       0.0|
|(346,[0,1,2,3,4,5...|  1.0|[1.66919385234539...|[0.35370932270187...|       1.0|
|(346,[0,1,2,3,4

In [113]:
#
future_match = matches_df.filter((matches_df["Match_Key"] == 4805) | (matches_df["Match_Key"] == 4800)
| (matches_df["Match_Key"] == 4801) | (matches_df["Match_Key"] == 4806) | (matches_df["Match_Key"] == 4804)
| (matches_df["Match_Key"] == 4799) | (matches_df["Match_Key"] == 4803) | (matches_df["Match_Key"] == 4798)
| (matches_df["Match_Key"] == 4802) | (matches_df["Match_Key"] == 4797) |
(matches_df["Match_Key"] == 7841) | (matches_df["Match_Key"] == 7840)
| (matches_df["Match_Key"] == 7837) | (matches_df["Match_Key"] == 7846) | (matches_df["Match_Key"] == 7842)
| (matches_df["Match_Key"] == 7844) | (matches_df["Match_Key"] == 7843) | (matches_df["Match_Key"] == 7838)
| (matches_df["Match_Key"] == 7845) | (matches_df["Match_Key"] == 7839)|
(matches_df["Match_Key"] == 12641) | (matches_df["Match_Key"] == 12643)
| (matches_df["Match_Key"] == 12644) | (matches_df["Match_Key"] == 12648) | (matches_df["Match_Key"] == 12645)
| (matches_df["Match_Key"] == 12639) | (matches_df["Match_Key"] == 12640) | (matches_df["Match_Key"] == 12642)
| (matches_df["Match_Key"] == 12646) | (matches_df["Match_Key"] == 12647) |
(matches_df["Match_Key"] == 2531) | (matches_df["Match_Key"] == 2528)
| (matches_df["Match_Key"] == 2526) | (matches_df["Match_Key"] == 2532) | (matches_df["Match_Key"] == 2524)
| (matches_df["Match_Key"] == 2527) | (matches_df["Match_Key"] == 2530) | (matches_df["Match_Key"] == 2529)
| (matches_df["Match_Key"] == 2525) |
(matches_df["Match_Key"] == 9991) | (matches_df["Match_Key"] == 9990)
| (matches_df["Match_Key"] == 9992) | (matches_df["Match_Key"] == 9987) | (matches_df["Match_Key"] == 9989)
| (matches_df["Match_Key"] == 9994) | (matches_df["Match_Key"] == 9988) | (matches_df["Match_Key"] == 9986)
| (matches_df["Match_Key"] == 9993))

In [114]:
future_match = future_match.withColumn("id", monotonically_increasing_id())
predictions = predictions.withColumn("id", monotonically_increasing_id())

In [115]:
future_match.show()

+---------+---------+---------------+----------+------------+------------+----------------+---------------+-------------+---------------+--------------+---------------+--------+--------+-------+-----------+-------+--------+---------+---------+------------+--------+--------+-----------+-------------------+-------------------+--------+--------+--------+---------------+---------------+-----------------------+----------------+--------------------+-----------------+-----------------+------------------+---------+---------------+----------+------------+------------+----------------+---------------+-------------+---------------+--------------+---------------+--------+--------+-------+-----------+-------+--------+---------+---------+------------+--------+--------+-----------+------------------+------------------+-------------------+--------+--------+---------------+---------------+-----------------------+----------------+--------------------+-----------------+-----------------+-----------------

In [116]:
predictions_completed = future_match.join(predictions, "id", "inner").drop("id")

In [117]:
predictions_completed = predictions_completed.withColumn("Result_Predict", when(predictions_completed["prediction"] == 0.0, "Win")
                                        .when(predictions_completed["prediction"] == 1.0, "Lose")
                                        .when(predictions_completed["prediction"] == 2.0, "Draw"))

In [118]:
display(predictions_completed.toPandas().head(5))

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Team_Onehot,Away_Team_Onehot,Features,Label,Features,Label,rawPrediction,probability,prediction,Result_Predict
0,4800,39,50.5,11.0,6.5,11.5,12.5,14.5,2.5,6.5,...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(50.5, 11.0, 6.5, 11.5, 12.5, 14.5, 2.5, 6.5, ...",0.0,"(50.5, 11.0, 6.5, 11.5, 12.5, 14.5, 2.5, 6.5, ...",0.0,"[2.7534308354999824, 1.0801213164214905, 1.449...","[0.6853134808323204, 0.12858200226826433, 0.18...",0.0,Win
1,4802,55,53.0,7.5,7.5,22.5,18.5,20.5,0.5,10.5,...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(53.0, 7.5, 7.5, 22.5, 18.5, 20.5, 0.5, 10.5, ...",1.0,"(53.0, 7.5, 7.5, 22.5, 18.5, 20.5, 0.5, 10.5, ...",1.0,"[1.6198549907937405, 1.8141156354068022, 1.449...","[0.3270021080137222, 0.3971155363956653, 0.275...",1.0,Lose
2,12639,126,42.5,9.0,4.5,22.0,17.5,20.5,3.0,4.5,...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(42.5, 9.0, 4.5, 22.0, 17.5, 20.5, 3.0, 4.5, 1...",1.0,"(42.5, 9.0, 4.5, 22.0, 17.5, 20.5, 3.0, 4.5, 1...",1.0,"[1.422895393956375, 1.864774509831114, 1.44986...","[0.279098682156636, 0.43417363496122524, 0.286...",1.0,Lose
3,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(47.0, 16.5, 6.0, 20.0, 19.0, 20.0, 3.0, 8.0, ...",0.0,"(47.0, 16.5, 6.0, 20.0, 19.0, 20.0, 3.0, 8.0, ...",0.0,"[1.3990684822374537, 1.8579556002320683, 1.449...","[0.27514716791886956, 0.435368687064164, 0.289...",1.0,Lose
4,2524,20,49.5,12.5,7.0,24.0,11.5,7.5,1.5,5.0,...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(49.5, 12.5, 7.0, 24.0, 11.5, 7.5, 1.5, 5.0, 1...",1.0,"(49.5, 12.5, 7.0, 24.0, 11.5, 7.5, 1.5, 5.0, 1...",1.0,"[1.7853937190013363, 1.7139486826087138, 1.449...","[0.37792810348235095, 0.3518689969984658, 0.27...",0.0,Win


In [119]:
predictions_completed = predictions_completed.drop(
    *["Features", "Label", "Home_Team_Onehot", "Away_Team_Onehot",
      "probability", "rawPrediction", "Away_Team_Index", "Home_Team_Index"]
    )

In [120]:
display(predictions_completed.toPandas().head(5))

,Match_Key,Home_Team,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,Home_Aerials_Won,Home_Clearances,Home_Offsides,Home_Goal_Kicks,...,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score,Match_Date,Result,prediction,Result_Predict
0,4800,39,50.5,11.0,6.5,11.5,12.5,14.5,2.5,6.5,...,79.444444,75.666667,72.000000,74.769231,2.0,1.0,20240519.0,Win,0.0,Win
1,4802,55,53.0,7.5,7.5,22.5,18.5,20.5,0.5,10.5,...,73.615385,78.625000,76.444444,70.555556,2.0,4.0,20240519.0,Lose,1.0,Lose
2,12639,126,42.5,9.0,4.5,22.0,17.5,20.5,3.0,4.5,...,70.625000,77.000000,76.166667,73.000000,2.0,3.0,20240523.0,Lose,1.0,Lose
3,12646,136,47.0,16.5,6.0,20.0,19.0,20.0,3.0,8.0,...,70.083333,77.100000,74.900000,77.266667,2.0,1.0,20240526.0,Win,1.0,Lose
4,2524,20,49.5,12.5,7.0,24.0,11.5,7.5,1.5,5.0,...,69.900000,71.500000,73.100000,72.857143,0.0,3.0,20240519.0,Lose,0.0,Win


In [121]:
# predictions_completed.write.csv("gs://football-data-etl/football-data-predicting/predictions_matches.csv", header=True)
predictions_completed.toPandas().to_csv('./predictions_matches.csv', index=False)

In [122]:
X_test.toPandas().to_csv('/content/drive/MyDrive/Colab Notebooks/football-data-predicting/X_test.csv', index=False)

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/Colab Notebooks/football-data-predicting'

In [ ]:
predictions.toPandas().to_csv('/content/drive/MyDrive/Colab Notebooks/football-data-predicting/predictions.csv', index=False)

In [ ]:
predictions_completed.toPandas().to_csv('/content/drive/MyDrive/Colab Notebooks/football-data-predicting/predictions_completed.csv', index=False)